# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Prabhaditya003/FlyRank.ai-internship-work-week1/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.inspection import permutation_importance

# Fix random seed for basic reproducibility as required
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Load the starter dataset
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit

# Fix random seed
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# 1. Flexible File Path Handler
possible_paths = [
    'content_refresh_anonymized.csv',
    '../data/raw/content_refresh_anonymized.csv',
    'data/raw/content_refresh_anonymized.csv'
]

file_path = next((p for p in possible_paths if os.path.exists(p)), None)

if file_path is None:
    try:
        from google.colab import files
        uploaded = files.upload()
        file_path = 'content_refresh_anonymized.csv'
    except ImportError:
        raise FileNotFoundError("Could not find 'content_refresh_anonymized.csv'.")

df = pd.read_csv(file_path)

# 2. Derive the target label (Label Trap fix)
# is_declining_label is derived from trend_direction == 'down'
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

# 3. Address rate column percentages (convert to decimals)
rate_cols = ['ctr', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct']
for col in rate_cols:
    if col in df.columns:
        df[col] = df[col] / 100.0

# 4. Handle missingness and avg_position gotcha
if 'avg_position' in df.columns:
    df['has_avg_position'] = (df['avg_position'] > 0).astype(int)
    df['avg_position'] = df['avg_position'].replace(0, np.nan)  # 0 means "no data"

if 'word_count' in df.columns:
    df['has_word_count'] = df['word_count'].notnull().astype(int)
    df['word_count'] = df['word_count'].fillna(0)

# 5. Drop leakages and metadata columns
# trend_pct and trend_direction are leakages and MUST NOT be used as features
leakage_cols = ['trend_pct', 'trend_direction']
non_feature_cols = ['is_declining_label', 'content_id', 'provider_used', 'model_used'] + leakage_cols

features = df.drop(columns=[c for c in non_feature_cols if c in df.columns], errors='ignore')

# 6. Separate client_id for grouping & encode features
client_ids = features['client_id']
X_raw = features.drop(columns=['client_id'])

X = pd.get_dummies(X_raw, drop_first=True)
X = X.fillna(X.median(numeric_only=True))

y = df['is_declining_label']
groups = client_ids

# 7. Grouped Split by client_id
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_SEED)
train_idx, test_idx = next(gss.split(X, y, groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# 1. Baseline Model (Simple Logistic Regression on just 2 features)
# Assuming CTR and word_count as a naive baseline rule
base_features = ['ctr', 'word_count']
X_train_base = X_train[[col for col in base_features if col in X_train.columns]].fillna(0)
X_test_base = X_test[[col for col in base_features if col in X_test.columns]].fillna(0)

baseline_model = LogisticRegression(random_state=RANDOM_SEED)
baseline_model.fit(X_train_base, y_train)
y_pred_base = baseline_model.predict(X_test_base)

# 2. Main Model (Random Forest)
rf_model = RandomForestClassifier(max_depth=6, random_state=RANDOM_SEED)
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)

# 3. Comparison Table (Non-negotiable)
results = {
    "Model": ["Base Rate (All No)", "Baseline (LogReg Rule)", "Random Forest (max_depth=6)"],
    "Precision": [
        0.0,
        precision_score(y_test, y_pred_base, zero_division=0),
        precision_score(y_test, y_pred_rf, zero_division=0)
    ],
    "Recall": [
        0.0,
        recall_score(y_test, y_pred_base, zero_division=0),
        recall_score(y_test, y_pred_rf, zero_division=0)
    ],
    "F1 Score": [
        0.0,
        f1_score(y_test, y_pred_base, zero_division=0),
        f1_score(y_test, y_pred_rf, zero_division=0)
    ]
}

comparison_table = pd.DataFrame(results)
display(comparison_table.round(3))

,Model,Precision,Recall,F1 Score
0,Base Rate (All No),0.000,0.000,0.000
1,Baseline (LogReg Rule),0.522,0.838,0.643
2,Random Forest (max_depth=6),0.633,0.789,0.703


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Feature Importance via Permutation
perm_importance = permutation_importance(rf_model, X_test, y_test, n_repeats=5, random_state=RANDOM_SEED)
sorted_idx = perm_importance.importances_mean.argsort()[-5:][::-1] # Top 5

print("--- Top 5 Features (Permutation Importance) ---")
for idx in sorted_idx:
    print(f"{X_test.columns[idx]}: {perm_importance.importances_mean[idx]:.4f}")

# Read the errors: Show 3 concrete wrong cases
# Find False Positives (Model predicted decline, but it didn't)
test_results = X_test.copy()
test_results['Actual'] = y_test
test_results['Predicted'] = y_pred_rf

errors = test_results[(test_results['Actual'] == 0) & (test_results['Predicted'] == 1)]

print("\n--- 3 Concrete Wrong Cases (False Positives) ---")
display(errors.head(3))

# Note on why they are hard:
print("\nError Analysis Interpretation:")
print("Look at the specific values in the 3 rows above. If 'scroll_rate' is high but 'ctr' is low, the model might over-index on poor search visibility despite good on-page engagement. These mixed signals create difficult boundary cases.")

--- Top 5 Features (Permutation Importance) ---
impressions_prev_30d: 0.1109
impressions_last_30d: 0.0403
clicks_last_30d: 0.0131
sessions_last_30d: 0.0067
ctr: 0.0027

--- 3 Concrete Wrong Cases (False Positives) ---


,search_volume,competition,cpc,word_count,char_count,impressions_90d,clicks_90d,pageviews_90d,sessions_90d,users_90d,...,char_count_tier_<8000,impression_tier_good,impression_tier_low,impression_tier_moderate,position_tier_page_1,position_tier_page_3_5,position_tier_striking,position_tier_top_3,Actual,Predicted
13,10.0,0.0,0.0,1342.0,8469.0,307,0,4,4,4,...,False,False,False,True,False,True,False,False,0,1
26,0.0,0.0,0.0,2686.0,17181.0,2426,3,9,9,9,...,False,False,False,True,False,True,False,False,0,1
36,0.0,0.0,0.0,2510.0,15518.0,371,5,6,5,5,...,False,False,False,True,True,False,False,False,0,1



Error Analysis Interpretation:
Look at the specific values in the 3 rows above. If 'scroll_rate' is high but 'ctr' is low, the model might over-index on poor search visibility despite good on-page engagement. These mixed signals create difficult boundary cases.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.